# Chương 1 — Milestone 1: Data Crawling

## 1.1. Thu thập dữ liệu và tính tái lập

**Discovery:** truy vấn arXiv theo ID và lưu metadata cấp paper. **Source:** tải riêng từng revision `vN`, giữ lại TeX/Bib, đánh dấu PDF-only và kiểm tra hash trong audit. **References:** Semantic Scholar cung cấp metadata cấp publication; không thể tự động gán citations cho từng revision lịch sử.

Dữ liệu thô và kết quả chạy lớn lưu ngoài GitHub. Tất cả giá trị trình bày bên dưới phải đến từ đầu ra có provenance của quá trình thu thập/kiểm định; số liệu thiếu phải hiển thị là *chưa có bằng chứng*, không thay bằng 0.

## 1.2. Mức độ bao phủ và trạng thái nguồn

Bảng, biểu đồ được tạo bằng Python **khi build Jupyter Book** từ bản tóm tắt `m1_summary.json`. Bản này do pipeline xuất sau khi chạy audit; có thể được đặt tại `book/_data/m1_summary.json` hoặc chỉ định qua biến môi trường `M1_SUMMARY_JSON`. Đây là dữ liệu *theo phạm vi đã kiểm định*, không tự động là kết quả toàn nhóm cho tới khi Duy bàn giao dữ liệu tổng hợp.


In [ ]:
import sys
from pathlib import Path

# Chạy được cả khi notebook được Jupyter Book execute từ thư mục book/.
book_dir = Path.cwd().resolve()
if not (book_dir / '_m1_report.py').exists():
    book_dir = book_dir / 'book'
if str(book_dir) not in sys.path:
    sys.path.insert(0, str(book_dir))
from _m1_report import load_verified_summary, summary_path, SOURCE_KEYS

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

report_path = summary_path()
summary = load_verified_summary(report_path)
if summary is None:
    display(Markdown('**Đang chờ summary M1 đã kiểm định.** Không tạo biểu đồ hoặc số liệu giả. '
                     'Đọc `docs/m1-book-handoff.md` để cung cấp đúng đầu vào.'))
else:
    coverage = summary['coverage']
    statuses = summary['source_statuses']
    display(Markdown(f"**Nguồn:** `{report_path.name}` | **Thời điểm:** "
                     f"{summary.get('generated_at', 'không rõ')} | **Audit:** "
                     f"{summary.get('integrity_status', 'không rõ')}"))
    rows = [
        ('Metadata hợp lệ', coverage['valid_metadata_papers']),
        ('Source version dự kiến', coverage['expected_versions']),
        ('Source version đã xác minh', coverage['verified_source_versions']),
        ('Độ phủ phiên bản (%)', coverage['source_coverage_pct']),
        ('Paper có raw references xác minh', coverage['raw_reference_papers_verified']),
    ]
    display(pd.DataFrame(rows, columns=['Chỉ số', 'Giá trị']))
    status_labels = {
        'verified_success': 'TeX verified',
        'verified_pdf_only': 'PDF-only verified',
        'pending': 'Pending',
        'failed': 'Failed',
        'not_found': 'HTTP 404',
        'conflicts': 'Conflict',
        'missing_files': 'Missing files',
    }
    status_frame = pd.DataFrame([
        {'Trạng thái': status_labels[k], 'Số version': statuses[k]} for k in SOURCE_KEYS
    ])
    ax = status_frame.plot.bar(x='Trạng thái', y='Số version', legend=False,
                               figsize=(9, 4), title='Kiểm định source theo phiên bản')
    ax.set_ylabel('Số version')
    ax.tick_params(axis='x', labelrotation=35)
    plt.tight_layout()
    plt.show()
    sizes = summary.get('source_file_statistics')
    if sizes is not None:
        display(pd.DataFrame(list(sizes.items()), columns=['Chỉ số kích thước', 'Giá trị']))
    else:
        display(Markdown('**Thống kê kích thước:** chưa có CSV source-version đã được audit xác nhận là mới nhất.'))


## 1.3. Phân tích corpus toàn nhóm — đang chờ Duy

Notebook EDA của Duy là nguồn phân tích cho phần này: phân bố số version/paper, tỷ lệ TeX so với PDF-only, tình trạng thiếu hoặc failed, phân bố raw/arXiv references và các so sánh theo dải ID. Không tổng hợp bằng phép cộng đơn giản khi các dải trùng nhau hoặc khác phạm vi; Duy cần xác nhận số ID duy nhất trước khi báo cáo toàn nhóm.

**Hợp đồng bàn giao:** `docs/m1-book-handoff.md`. Khi nhận notebook và file số liệu từ Duy, Tuấn sẽ review phương pháp, tích hợp các cell vẽ bảng/hình tại lúc build và bổ sung kết luận có bằng chứng. Không tự tạo số liệu hoặc ảnh tĩnh trong khi chờ.

## 1.4. Benchmark và giới hạn diễn giải

Benchmark phải phản ánh số đo thực tế trên Colab CPU-only: wall time, RAM trung bình/cực đại, peak disk, dung lượng đầu ra. Nhiều batch độc lập không đồng nghĩa một phép đo end-to-end duy nhất, và không được sửa các con số đo để cộng đủ 5.000 bài. Source PDF-only, lỗi mạng, HTTP 404 phải phân loại riêng, không gộp thành TeX thành công.


In [ ]:
if summary is None:
    display(Markdown('**Benchmark:** chưa có summary đã xác minh.'))
else:
    evidence = summary['benchmark_evidence']
    official = evidence.get('official_version_pipeline_metrics')
    if official is None:
        display(Markdown('**Benchmark end-to-end:** chưa đủ bằng chứng để công bố số đo chính thức.'))
    else:
        display(pd.DataFrame(list(official.items()), columns=['Chỉ số', 'Giá trị']))
    for requirement in summary['missing_evidence']:
        display(Markdown(f'- **Thiếu bằng chứng:** {requirement}'))
